In [ ]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("run_workflow_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [ ]:
start_time = time.time()  

logger.info("Notebook execution started.")

In [ ]:
# === Standard Library Imports ===
import os
import sys
import logging
import warnings
from datetime import datetime
from pathlib import Path
from typing import List

# === Third-Party Imports ===
import mlflow
import torch
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.llms import LlamaCpp
from langchain_core.callbacks import CallbackManager, StreamingStdOutCallbackHandler
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline, HuggingFaceEndpoint
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda, RunnableMap
from langchain.schema.document import Document
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# NOTE: Galileo dependencies have been removed for public release
# import promptquality as pq
# import galileo_protect as gp
# from galileo_protect import ProtectTool, ProtectParser, Ruleset

# Define the relative path to the 'src' directory (one level up from current working directory)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

# === Project-Specific Imports (from src.utils) ===
from src.utils import (
    load_config_and_secrets,
    configure_proxy,
    initialize_llm,
    setup_galileo_environment,
    initialize_galileo_evaluator,
    initialize_galileo_protect,
    initialize_galileo_observer,
    configure_hf_cache,
)
from src.prompt_templates import format_rag_chatbot_prompt

# === Import the service class ===
from core.chatbot_service.chatbot_service import ChatbotService

In [ ]:
CONFIG_PATH = "../configs/config.yaml"
SECRETS_PATH = "../configs/secrets.yaml"
DATA_PATH = "../data"

# NOTE: Galileo project names preserved for compatibility but functionality disabled
GALILEO_EVALUATE_PROJECT_NAME = "AIStudio-Chatbot-EvaluateProject"
GALILEO_PROTECT_PROJECT_NAME = "AIStudio-Chatbot-ProtectProject" 
GALILEO_OBSERVE_PROJECT_NAME = "AIStudio-Chatbot-ObserveProject" 
GALILEO_EVALUATE_AND_PROTECT_PROJECT_NAME = "AIStudio-Chatbot-EvaluateProtectProject"

# Local model path configuration
LOCAL_MODEL_PATH = "/home/jovyan/local/datafabric/meta-llama3.1-8b-Q8.gguf"

# MLflow configuration
MLFLOW_EXPERIMENT_NAME = "AIStudio-Chatbot-Experiment"
MLFLOW_RUN_NAME = "AIStudio-Chatbot-Run"
MLFLOW_MODEL_NAME = "AIStudio-Chatbot-Model"

# Demo folder configuration
DEMO_FOLDER = "../demo"

# Model Service 

In this section, we demonstrate how to deploy a RAG-based chatbot service with integrated monitoring capabilities. This service provides a REST API endpoint that allows users to query the knowledge base with natural language questions, upload new documents to the knowledge base, and manage conversation history. This service encapsulates all the functionality we developed in this notebook, including the document retrieval system and RAG-based question answering capabilities. It demonstrates how to use our ChatbotService from the src/service directory.

**Note**: Galileo integration has been removed for public release. The service maintains full functionality for document retrieval and Q&A without proprietary dependencies.

In [ ]:
mlflow.set_tracking_uri('/phoenix/mlflow')
# === Set MLflow experiment context ===
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

# === Validate local model file path ===
if not os.path.exists(LOCAL_MODEL_PATH):
    logger.info(f"⚠️ Warning: Model file not found at {LOCAL_MODEL_PATH}. Please verify the path.")

# === Log and register model to MLflow ===
with mlflow.start_run(run_name=MLFLOW_RUN_NAME) as run:
    
    # Log model artifacts using custom ChatbotService
    ChatbotService.log_model(
        artifact_path=MLFLOW_MODEL_NAME,
        config_path=CONFIG_PATH,
        secrets_path=SECRETS_PATH,
        docs_path=DATA_PATH,
        model_path=LOCAL_MODEL_PATH,
        demo_folder=DEMO_FOLDER
    )

    # Construct the URI for the logged model
    model_uri = f"runs:/{run.info.run_id}/{MLFLOW_MODEL_NAME}"

    # Register the model into MLflow Model Registry
    mlflow.register_model(
        model_uri=model_uri,
        name=MLFLOW_MODEL_NAME
    )

    logger.info(f"✅ Model registered successfully with run ID: {run.info.run_id}")

In [ ]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).